# EuroSAT RGB vs multispectral — Colab driver

Runs the full experiment matrix on a free **T4 GPU** (~5-7 GPU-hours). Idempotent and resumable: if the session drops, just re-run from the top — completed runs skip themselves.

**Before running:** Runtime -> Change runtime type -> GPU (T4). Then Runtime -> Run all.

Set `GITHUB_REPO` below to your repository URL (and add a token for a private repo).

In [ ]:
#@title 1. Clone the code repository
GITHUB_REPO = "https://github.com/Mohammed-AlKuhali/eurosat-ms-classification.git"  #@param {type:"string"}
import os
if not os.path.isdir('eurosat-ms-classification'):
    !git clone $GITHUB_REPO
%cd eurosat-ms-classification
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Install dependencies
!pip install -q -e . >/dev/null
!pip install -q torch torchvision timm==1.0.11 torchgeo==0.6.2 grad-cam==1.5.4 statsmodels scikit-image >/dev/null
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
#@title 3. Mount Drive and download the dataset (cached after first run)
from google.colab import drive
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/eurosat_cache'
!python scripts/download_data.py --root data/raw --cache $CACHE
os.environ['EUROSAT_DATA_ROOT'] = 'data/raw/EuroSAT_MS'
# The split manifests and train stats are committed in the repo; regenerate only
# if you want to re-verify them (deterministic — produces identical files).
# !python scripts/make_manifests.py

In [ ]:
#@title 4. (Optional) Mirror results to Drive so they survive a disconnect
RESULTS_DRIVE = '/content/drive/MyDrive/eurosat_results'
!mkdir -p $RESULTS_DRIVE && rm -rf results && ln -s $RESULTS_DRIVE results
!mkdir -p results/metrics results/predictions results/figures checkpoints
# Re-link checkpoints to Drive too (so partial progress survives):
!rm -rf checkpoints && ln -s $RESULTS_DRIVE/checkpoints checkpoints && mkdir -p checkpoints

In [ ]:
#@title 5. Run the full matrix (priority order; resumable)
import os
os.environ['DEVICE'] = 'cuda'
!bash scripts/run_all.sh

## If a session times out
Re-run cells 1-5. Completed arm/seed/fraction runs are detected from `results/metrics/*.json` and skipped, so each new session only does the remaining work. With results symlinked to Drive (cell 4), nothing is lost.

## After all runs complete
Commit the `results/` folder back to the repo (metrics + prediction CSVs are small), then run the analysis notebook locally:
```bash
git add results && git commit -m 'Add Colab experiment results'
```